In [57]:
import pandas as pd
import numpy as np

from pathlib import Path

In [58]:
exam_result_path = Path("/Users/abelluc/FAUbox/DigiKolleg/BPOS/Exam")

In [59]:
exam_results = pd.read_excel(exam_result_path.joinpath("2024-02-05_BPOS_Exam_Results.xlsx"))
exam_results.dropna(subset=["Login"],inplace=True, how="all")
exam_results.head()

In [60]:
exam_results.set_index("Login", inplace=True)
exam_results.index.names = ["student_id"]
exam_results.head()

In [61]:
points = exam_results["Test Results in Points"]
points.hist(bins=45)

In [62]:
points = pd.DataFrame(points)
points.columns = ["points"]
points.head()

In [63]:
pass_score = 45.0
max_score = 90.0
bin_size = round((max_score-pass_score) / 11, 1)
bin_size

In [64]:
grades = [1.0, 1.33, 1.66, 2.0, 2.33, 2.66, 3.0, 3.33, 3.66, 4.0, 4.33, 5.0]
grades = grades[::-1]

In [65]:
min_points = [pass_score + bin_size * i for i in range(-2,len(grades)-1)]
min_points

In [66]:
grade_map = pd.DataFrame(zip(min_points, grades), columns=["min_points", "grade"])
grade_map["max_points"] = grade_map["min_points"] + bin_size - 0.1
grade_map.set_index("grade", inplace=True)
grade_map.loc[1.0, "max_points"] = max_score
grade_map.loc[5.0, "min_points"] = 0.0

grade_map.sort_index(inplace=True)

grade_map

In [67]:
# return the value of the index of grade_map where points is between min_points and max_points
points["grade"] = [grade_map[(grade_map["min_points"] <= point) & (grade_map["max_points"] >= point)].index[0] for point in points["points"]]
points


In [68]:
# Without Bonus

points["grade"].value_counts().sort_index().plot(kind="bar")

In [69]:
points["grade"].describe()

In [70]:
bonus_points = pd.read_excel("/Users/abelluc/FAUbox/DigiKolleg/BPOS/Exercise/WS23_24_Results/result_final.xlsx")
#rename column idm to identifier 
bonus_points.rename(columns={"idm":"student_id"}, inplace=True)
bonus_points.set_index("student_id", inplace=True)
bonus_points

In [71]:
# join dataframes
points = points.join(bonus_points, how="left")
points.fillna(0, inplace=True)

#points.drop("student_id", axis=1, inplace=True)
points

In [72]:
points.drop("passed", axis=1, inplace=True)

In [73]:
points

In [74]:
points["grade_with_bonus"] = points["grade"] - points["bonus"]

# set grade_with_bonus to 1.0 if grade_with_bonus is smaller than 1.0
points["grade_with_bonus"][points["grade_with_bonus"] < 1.0] = 1.0

# set grade_with_bonus to grade value if grade is greater than 4.0
points["grade_with_bonus"][points["grade"] > 4.0] = points["grade"]

points

In [75]:
points = round(points, 1)
points.sort_values("grade", inplace=True)
points

In [76]:
points.describe()

In [77]:
points["grade_with_bonus"].value_counts().sort_index().plot(kind="bar")

In [78]:
# save to csv
points.to_csv(exam_result_path.joinpath("2023-02_BPOS_Exam_results_final.csv"))